In [1]:
import os 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
import seaborn as sns
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA

In [2]:
PREFIX_MAP = {
  "Mouse-2023": "ENSMUS",
  "Zebrafish-2024": "ENSDARG",
  "Drosophila-2025": "FBgn"
}

# Config
DATASET = "Zebrafish-2024"
GENE_PREFIX = PREFIX_MAP.get(DATASET)
PROCESSED_DIR = os.path.join(DATASET, "processed-data")
QC_DIR = os.path.join(DATASET, "QC-results")

counts_path = os.path.join(PROCESSED_DIR, "counts.csv")
metadata_path = os.path.join(PROCESSED_DIR, "metadata.csv")
qc_metrics_path = os.path.join(PROCESSED_DIR, "qc_metrics.csv")


**Reviewing Flagged Samples**

In [3]:
qc_df = pd.read_csv(qc_metrics_path)
flagged = qc_df.loc[(qc_df['absolute_thresholds'] != 'Pass') | (qc_df['relative_thresholds'] != 'Pass')]
flagged_path = os.path.join(QC_DIR, "flagged_samples.csv")
flagged.to_csv(flagged_path, index=False, header=True)

**Normality Check**

In [5]:
qc_df = pd.read_csv(qc_metrics_path)
metrics = ['total_reads', 'mapping_rate', 'n_detected_genes', 'multimapping_rate']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

# Each metric gets its own subplot (axis)
for i, met in enumerate(metrics):
  ax = axes[i]
  # Calculate density histogram
  # Density = True changes y-axis from number of samples to probability density
  data = qc_df[met].dropna()
  ax.hist(data, bins=10, density=True, alpha=0.6, color='skyblue', edgecolor='black', label='Actual Data')
  
  # Fit a normal distribution to the data (Mean and Std Dev)
  mu, std = norm.fit(data)
  
  # Create the theoretical normal distribution curve (PDF)
  xmin, xmax = ax.get_xlim()
  x = np.linspace(xmin, xmax, 100)
  p = norm.pdf(x, mu, std)
  
  # Overlay the normal curve
  ax.plot(x, p, 'r', linewidth=2, label=rf'Normal Fit\n$\mu$={mu:.2g}\n$\sigma$={std:.2g}')
  
  ax.set_title(f"Normality Check: {met}")
  ax.set_xlabel("Value")
  ax.set_ylabel("Density")
  ax.legend()

plt.tight_layout()
plt.savefig(os.path.join(QC_DIR, "normality_plots.png"))
plt.close()

**Data Tranformation**

In [3]:
def transform_counts(counts_path):
  '''
  Consumes a path to a csv counts file where the index is the first column and 
  performs median-of-ratios normalization and log2 compression, returning a 
  dataframe of the transformed counts. 

  Returns: DataFrame 
  '''
  # Reading in and filtering count data
  counts = pd.read_csv(counts_path, index_col=0)
  f_counts = counts.loc[:, counts.columns.astype(str).str.startswith(GENE_PREFIX)]
  f_counts = f_counts.loc[:, f_counts.sum(axis=0) >= 10] # min count filter

  # Calculate size factors using geometric mean
  log_x = np.log(f_counts.astype(float).replace(0, np.nan))
  log_gm = np.nanmean(log_x.values, axis=0)
  gm = np.exp(log_gm)
  gm[~np.isfinite(gm)] = np.nan
  ratios = f_counts.values / gm
  sf = np.nanmedian(ratios, axis=1)

  # Transform (normalize and stabilize variance)
  t_counts = np.log2(f_counts.div(sf, axis=0) + 1.0)
  return t_counts

**Correlation Heatmap**

In [16]:
def plot_heatmap(counts, method, postF):
  '''
  Consumes a DataFrame counts, a string method, and a string outpath and 
  generates a heatmap using average clustering with the method specified. 

  Parameters: 
  - Counts (DataFrame): samples x genes dataframe of transformed counts
  - Method (Str): correlation method ('pearson', 'spearman', or 'kendall')
  - PostF (Bool): If the data is filtered (True) or not (False)


  Returns: None

  Effects: 
  - saves a heatmap of name "heatmap_{method}.png" to DATASET/QC-results/
  '''
  # prep data for heatmap 
  corr = counts.T.corr(method=method) # 2D n x n square matrix of pairwise correlations between samples
  dist = 1 - corr # 2D n x n square matrix of pairwise distances (1 - correlation) between samples
  condensed_dist = squareform(dist, checks=False) # 1D vector of non-redundant pairwise distances for clustering
  linkage = hierarchy.linkage(condensed_dist, method='average') # Clustering tree (dendogram) based on distances

  # heatmap with dendrogram
  plt.figure(figsize=(12, 10))
  g = sns.clustermap(dist,
                     row_linkage=linkage, 
                     col_linkage=linkage,
                     cmap='RdYlBu_r', 
                     figsize=(12,10),
                     cbar_kws={'label': f'{method.capitalize()} Distance (1 - corr)'},
                     xticklabels=counts.index,
                     yticklabels=counts.index)
  g.ax_col_dendrogram.set_title("Sample Distance Matrix", pad =20, fontsize=20)
  g.ax_cbar.set_position((0.05, 0.83, 0.03, 0.2))
  plt.close()
  if postF:
    outpath = os.path.join(QC_DIR, f"filtered_heatmap_{method}.png")
  else:
    outpath = os.path.join(QC_DIR, f"heatmap_{method}.png")
  g.savefig(outpath)

In [17]:
def plot_meta_heatmap(counts, method, postF):
  '''
  Consumes a DataFrame counts, a string method, and a string outpath and 
  generates a heatmap using average clustering with the method specified. Plots
  the difference of differences.  

  Parameters: 
  - Counts (DataFrame): samples x genes dataframe of transformed counts
  - Method (Str): correlation method ('pearson', 'spearman', or 'kendall')
  - PostF (Bool): If the data is filtered (True) or not (False)

  Returns: None

  Effects: 
  - saves a heatmap of name "meta-heatmap-{method}.png" to DATASET/QC-results/
  '''
  # prep data for heatmap 
  corr = counts.T.corr(method=method)
  dist = 1 - corr

  # heatmap with dendrogram
  plt.figure(figsize=(12, 10))
  g = sns.clustermap(dist,
                    method='average',
                    cmap='RdYlBu_r', 
                    figsize=(12,10),
                    cbar_kws={'label': f'{method.capitalize()} Distance (1 - corr)'},
                    xticklabels=counts.index,
                    yticklabels=counts.index)
  g.ax_col_dendrogram.set_title("Sample Distance Matrix", pad =20, fontsize=20)
  g.ax_cbar.set_position((0.05, 0.83, 0.03, 0.2))
  plt.close()
  if postF:
    outpath = os.path.join(QC_DIR, f"filtered_meta_heatmap_{method}.png")
  else:
    outpath = os.path.join(QC_DIR, f"meta_heatmap_{method}.png")
  g.savefig(outpath)


**PCA**

In [26]:
def plot_pca(counts, metadata, n_PCs, n_genes, pc_x, pc_y, color_by, shape_by, postF):
    '''
    Consumes DataFrames of counts and metadata, as well as configuration parameters
    and generates a PCA plot and Scree plot. 
    counts: DataFrame
    metadata: DataFrame
    n_PCs: Int, n_genes: Int, pc_x: String (e.g. 'PC1'), pc_y: String (e.g. 'PC2')
    color_by: One of columns in metadata, shape_by: One of columns in metadata
    postF: Bool

    Returns: None, but generates 2 plots
    '''
    # Get subset of most-variable genes
    gene_variance = counts.var(axis=0) # Calculate var for each gene (column)
    most_var_genes_index = gene_variance.sort_values(ascending=False).head(n_genes).index
    most_var_genes = counts[most_var_genes_index]
    
    # PCA Calculation
    pca = PCA(n_components=n_PCs)
    pca_results = pca.fit_transform(most_var_genes) 
    pc_vars = {f'PC{i+1}': var for i, var in enumerate(pca.explained_variance_ratio_)}
    
    pca_df = pd.DataFrame(
        data=pca_results, 
        columns=[f'PC{i+1}' for i in range(n_PCs)], 
        index=counts.index
    )
    pca_df = pca_df.join(metadata)
    
    # PCA Plotting
    colors = "bright"
    plt.figure(figsize=(10, 7))
    ax = sns.scatterplot(data=pca_df, x=pc_x, y=pc_y, hue=color_by, palette=colors, style=shape_by, s=100)
    
    # Remove legend headers (standard cleanup)
    handles, labels = ax.get_legend_handles_labels()
    headers = [color_by, shape_by] # Headers to remove
    new_handles = [h for h, l in zip(handles, labels) if l not in headers]
    new_labels = [l for h, l in zip(handles, labels) if l not in headers]
    ax.legend(handles=new_handles, labels=new_labels)
    
    # Label datapoints to identify outliers (Loop starts here)
    for i in range(pca_df.shape[0]):
        plt.text(
            x=pca_df[pc_x].iloc[i] + 0.1, 
            y=pca_df[pc_y].iloc[i] + 0.1, 
            s=str(pca_df.replicate.iloc[i]), 
            fontsize=10, 
            weight='bold',
            color='black'
        )
    
    # --- The following labels and save commands must be OUTSIDE the loop ---
    plt.xlabel(f"{pc_x} ({pc_vars[pc_x]*100:.1f}%)")
    plt.ylabel(f"{pc_y} ({pc_vars[pc_y]*100:.1f}%)")
    plt.title(f"{pc_x} vs {pc_y}: {color_by.capitalize()} and {shape_by.capitalize()} Effects ({n_genes} Genes)")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    
    if postF:
        pca_path = os.path.join(QC_DIR, f"filtered_{n_genes}_{pc_x}_{pc_y}_by_{color_by}.png")
    else:
        pca_path = os.path.join(QC_DIR, f"{n_genes}_{pc_x}_{pc_y}_by_{color_by}.png")
    plt.savefig(pca_path)
    plt.close()

    # --- Scree plot (Also outside the sample loop) ---
    exp_var_pct = pca.explained_variance_ratio_ * 100
    cum_var_pct = np.cumsum(exp_var_pct)
    
    plt.figure(figsize=(10, 6))
    x_axis = range(1, len(exp_var_pct) + 1)
    bars = plt.bar(x_axis, exp_var_pct, alpha=0.7, color='skyblue', label='Individual Variance')
    
    # Text labels on top of bars
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f'{yval:.1f}%', 
                 ha='center', va='bottom', fontsize=9)
    
    # Line plot for cumulative variance
    plt.plot(x_axis, cum_var_pct, marker='o', color='red', linestyle='--', label='Cumulative Variance')
    plt.text(
        x=x_axis[-1],
        y=cum_var_pct[-1] + 2,
        s=f"Total: {cum_var_pct[-1]:.1f}%", 
        color='red', fontweight='bold', ha='center'
    )
    plt.title(f"Scree Plot: Variance Explained by {n_PCs} Principal Components ({n_genes} Genes)")
    plt.xlabel("Principal Component")
    plt.ylabel("Percentage of Variance Explained (%)")
    plt.xticks(x_axis)
    plt.ylim(0, 105)
    plt.legend(loc='upper left')
    plt.grid(axis='y', linestyle='--', alpha=0.3)
    plt.tight_layout()
    if postF:
        scree_path = os.path.join(QC_DIR, f"filtered_{n_genes}_Scree.png")
    else:
        scree_path = os.path.join(QC_DIR, f"{n_genes}_Scree.png")
    plt.savefig(scree_path)
    plt.close()

**Generate Plots**

In [27]:
t_counts = transform_counts(counts_path)
metadata = pd.read_csv(metadata_path, index_col=0)

plot_heatmap(t_counts, "kendall", postF=False)
plot_meta_heatmap(t_counts, "kendall", postF=False)
plot_pca(t_counts, metadata, 6, 37992, 'PC1', 'PC2', "treatment", "replicate", postF=False)

c:\Users\ibanw\Downloads\preclinical_differential_expression\.venv\Lib\site-packages\seaborn\matrix.py:530: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  linkage = hierarchy.linkage(self.array, method=self.method,
c:\Users\ibanw\Downloads\preclinical_differential_expression\.venv\Lib\site-packages\seaborn\matrix.py:530: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  linkage = hierarchy.linkage(self.array, method=self.method,


<Figure size 1200x1000 with 0 Axes>

<Figure size 1200x1000 with 0 Axes>

**Outlier Removal**

In [24]:
counts = pd.read_csv(counts_path, index_col=False)
metadata = pd.read_csv(metadata_path, index_col=False)

outlier_map = {
  "Zebrafish-2024": ['CTRL_rep1', 'BMAA_rep3', 'BMAApure_rep5', 'BMAA_rep2']
  }

outlier_samples = outlier_map.get(DATASET)

filtered_counts = counts[~counts['sample'].isin(outlier_samples)]
filtered_counts_path = os.path.join(PROCESSED_DIR, "filtered_counts.csv")
filtered_counts.to_csv(filtered_counts_path, index=False)
filtered_metadata = metadata[metadata['sample'].isin(filtered_counts['sample'])]
filtered_metadata_path = os.path.join(PROCESSED_DIR, "filtered_metadata.csv")
filtered_metadata.to_csv(filtered_metadata_path, index=False)

**Regenerate Plots**

In [33]:
t_counts = transform_counts(filtered_counts_path)
metadata = pd.read_csv(filtered_metadata_path, index_col=0)

plot_heatmap(t_counts, "kendall", postF=True)
plot_pca(t_counts, metadata, 6, 500, 'PC2', 'PC3', "treatment", "replicate", postF=True)

<Figure size 1200x1000 with 0 Axes>